In [1]:
import pandas as pd

In [2]:
df=pd.read_csv(r"D:\Python Projects\CourseCompass.csv")

In [3]:
df

,course_id,course_name,category,degree_level,duration_years,eligible_streams,key_subjects,entrance_exams,interest_tags,career_paths,start_salary_lpa_min,start_salary_lpa_max,difficulty_1_5,description
0,C025,B.Sc Agriculture,Agriculture,UG,4.0,PCB|PCM|PCMB,Biology|Chemistry,CUET (ICAR AIEEA UG)|State agri univ counselli...,farming|nature|environment|research,Agri Officer|Agri-business Manager|Farm Entrep...,3,8,3,"Crop science, soil, agri-economics and farm ma..."
1,C020,B.Sc Nursing,Allied Health,UG,4.0,PCB|PCMB,Biology|Chemistry,State merit counselling|Institute entrance,healthcare|helping people|patient care,Staff Nurse|Nurse Educator|Overseas Nursing,3,8,3,Clinical nursing; strong demand in India and a...
2,C021,B.Pharm Pharmacy,Allied Health,UG,4.0,PCM|PCB|PCMB,Chemistry|Biology|Maths,State merit counselling|BITSAT|VITEEE,chemistry|healthcare|lab work|industry,Pharmacist|Drug Inspector|Pharma R&D,3,7,3,"Drug formulation, pharmacology and pharma indu..."
3,C022,BPT Physiotherapy,Allied Health,UG,4.5,PCB|PCMB,Biology|Physics,Merit|State counselling,healthcare|sports|rehabilitation,Physiotherapist|Sports Therapist,3,8,3,"Rehabilitation and movement science, includes ..."
4,C009,B.Arch Architecture,Architecture,UG,5.0,PCM,Maths|Physics|Drawing,NATA|JEE Main Paper 2,design|drawing|creativity|buildings,Architect|Urban Planner|Interior Designer,3,10,4,Design of buildings and spaces; needs strong d...
5,C032,BA Economics,Arts & Humanities,UG,3.0,ARTS|COM_MATHS|PCM,Economics|Maths,CUET|Merit,economics|policy|data|analysis,Economist|Policy Analyst|Banker|Civil Services...,3,9,3,"Micro/macro economics, statistics; maths stron..."
6,C033,BA English Literature,Arts & Humanities,UG,3.0,ARTS|COM_MATHS|COM_NOMATHS|PCM|PCB,English,CUET|Merit,reading|writing|communication|creativity,Content Writer|Teacher|Editor|Journalist,2,6,2,"Literature, linguistics and critical writing."
7,C034,BA History / Political Science,Arts & Humanities,UG,3.0,ARTS|COM_MATHS|COM_NOMATHS|PCM|PCB,History|Political Science,CUET|Merit,history|politics|civil services|debate,Civil Services Aspirant|Researcher|Teacher|Jou...,2,6,2,Popular base for UPSC/TNPSC preparation.
8,C035,BA/B.Sc Psychology,Arts & Humanities,UG,3.0,ARTS|COM_MATHS|COM_NOMATHS|PCB|PCM,Psychology|Biology,CUET|Merit,people|counselling|mental health|research,Counsellor|HR Executive|Clinical Psychologist ...,2,7,3,Human behaviour; clinical roles need a Master's.
9,C026,B.Com General,Commerce,UG,3.0,COM_MATHS|COM_NOMATHS|ARTS|PCM|PCB,Accountancy|Economics,CUET|Merit,finance|business|accounts|numbers,Accountant|Banker|Tax Executive|MBA aspirant,2,6,2,"Accounting, finance, taxation and business law."


In [4]:
print(df.isnull().sum())

course_id               0
course_name             0
category                0
degree_level            0
duration_years          0
eligible_streams        0
key_subjects            0
entrance_exams          0
interest_tags           0
career_paths            0
start_salary_lpa_min    0
start_salary_lpa_max    0
difficulty_1_5          0
description             0
dtype: int64


In [5]:
# Combine relevant text columns into a single search corpus for each course
df["combined_features"] = (
    df["interest_tags"].astype(str)
    + " "
    + df["career_paths"].astype(str)
    + " "
    + df["description"].astype(str)
)

# Preview the combined text of the first row
print(df["combined_features"].iloc[0])

farming|nature|environment|research Agri Officer|Agri-business Manager|Farm Entrepreneur Crop science, soil, agri-economics and farm management.


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer (removing common English stop words)
tfidf = TfidfVectorizer(stop_words="english")

# Fit and transform the combined features
tfidf_matrix = tfidf.fit_transform(df["combined_features"])

# Check the shape of your matrix (Rows = number of courses, Columns = unique vocabulary words)
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (43, 360)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
def get_recommendations(student_stream, student_interests, top_n=5):
  # Rule-based filtering: Check if the student's stream matches eligible_streams
  eligible_df = df[
      df["eligible_streams"].str.contains(student_stream, case=False, na=False)
      | df["eligible_streams"].str.contains("ANY", case=False, na=False)
  ].copy()

  if eligible_df.empty:
    return "No courses found for this stream."

  # Get indices of filtered courses
  eligible_indices = eligible_df.index.tolist()

  # Transform student's interest query
  query_vec = tfidf.transform([student_interests])

  # Calculate cosine similarity only for eligible courses
  subset_matrix = tfidf_matrix[eligible_indices]
  similarity_scores = cosine_similarity(query_vec, subset_matrix).flatten()

  # Add scores to the filtered dataframe
  eligible_df["match_score"] = similarity_scores

  # Sort by top scores
  recommendations = eligible_df.sort_values(
      by="match_score", ascending=False
  ).head(top_n)

  # Return useful columns
  return recommendations[
      [
          "course_id",
          "course_name",
          "degree_level",
          "duration_years",
          "career_paths",
          "match_score",
      ]
  ]

In [9]:
# Test with a Computer Science / PCM stream student
get_recommendations(
    student_stream="PCM",
    student_interests="coding, algorithms, software development, machine learning",
)

,course_id,course_name,degree_level,duration_years,career_paths,match_score
15,C001,B.E./B.Tech Computer Science & Engineering,UG,4.0,Software Engineer|Data Scientist|Cloud Enginee...,0.268068
34,C010,B.Sc Computer Science,UG,3.0,Software Developer|Web Developer|Tester,0.182929
16,C002,B.E./B.Tech Artificial Intelligence & Data Sci...,UG,4.0,ML Engineer|Data Analyst|AI Engineer,0.159405
11,C011,BCA Bachelor of Computer Applications,UG,3.0,Web Developer|App Developer|IT Support|MCA asp...,0.153961
18,C004,B.E./B.Tech Mechanical Engineering,UG,4.0,Design Engineer|Manufacturing Engineer|Automob...,0.094776


In [10]:
import pickle

# Save the dataframe
with open("course_compass_df.pkl", "wb") as f:
  pickle.dump(df, f)

# Save the TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
  pickle.dump(tfidf, f)

print("Models saved successfully using pickle!")

Models saved successfully using pickle!
